# Chương 4 - Phần 1: Khởi Tạo Mã Băm Wavelet Hashing

**Thông tin học viên:**
- **Họ và tên:** Nguyễn Ngọc Anh  
- **Lớp:** CN22H  
- **Môn học:** Thị giác Máy tính (Computer Vision)  

---

## 1. Import Thư Viện & Cài Đặt Ban Đầu

Import các thư viện và cấu hình cơ bản.

In [ ]:
import os
import json
import cv2
import numpy as np
import pywt
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 11
print("Đã import thành công các thư viện!")
print("Phiên bản PyWavelets:", pywt.__version__)
print("Phiên bản OpenCV:", cv2.__version__)

## 2. Chuẩn Bị Dữ Liệu Kiểm Thử

Chúng ta đọc 4 ảnh gốc và sinh ra các biến thể tương tự của `anh.jpg` (xoay ảnh, đổi độ sáng, thêm nhiễu) để tạo tập dữ liệu so khớp.

In [ ]:
img_dir = "../data/input"
os.makedirs(img_dir, exist_ok=True)
origin_path = os.path.join(img_dir, "anh.jpg")

if not os.path.exists(origin_path):
    raise FileNotFoundError(f"Không tìm thấy ảnh gốc tại: {origin_path}")

# Đọc ảnh gốc và chuyển ảnh xám
img_bgr = cv2.imread(origin_path)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
h, w = img_gray.shape

# 1. Ảnh xoay
M = cv2.getRotationMatrix2D((w/2, h/2), 15, 1.0)
img_rotated = cv2.warpAffine(img_gray, M, (w, h))
cv2.imwrite(os.path.join(img_dir, "anh_rotated.jpg"), img_rotated)

# 2. Ảnh tăng độ sáng
img_bright = np.clip(img_gray.astype(np.int16) + 40, 0, 255).astype(np.uint8)
cv2.imwrite(os.path.join(img_dir, "anh_bright.jpg"), img_bright)

# 3. Ảnh bị nhiễu Gaussian
noise = np.random.normal(0, 15, img_gray.shape).astype(np.int16)
img_noisy = np.clip(img_gray.astype(np.int16) + noise, 0, 255).astype(np.uint8)
cv2.imwrite(os.path.join(img_dir, "anh_noisy.jpg"), img_noisy)

# 4. Ảnh xoay mạnh (30 độ)
M30 = cv2.getRotationMatrix2D((w/2, h/2), 30, 1.0)
img_rotated_30 = cv2.warpAffine(img_gray, M30, (w, h))
cv2.imwrite(os.path.join(img_dir, "anh_rotated_30.jpg"), img_rotated_30)

# 5. Ảnh làm mờ (Blur nặng)
img_blurred = cv2.GaussianBlur(img_gray, (15, 15), 0)
cv2.imwrite(os.path.join(img_dir, "anh_blurred.jpg"), img_blurred)

# 6. Ảnh nhiễu Gaussian nặng (std = 40)
noise_heavy = np.random.normal(0, 40, img_gray.shape).astype(np.int16)
img_noisy_heavy = np.clip(img_gray.astype(np.int16) + noise_heavy, 0, 255).astype(np.uint8)
cv2.imwrite(os.path.join(img_dir, "anh_noisy_heavy.jpg"), img_noisy_heavy)

print("Đã sinh các biến thể ảnh tương tự thành công!")

In [ ]:
all_images = {
    "Gốc (anh.jpg)": img_gray,
    "Tương tự - Sáng hơn": img_bright,
    "Tương tự - Xoay 15 độ": img_rotated,
    "Tương tự - Nhiễu": img_noisy,
    "Tương tự - Xoay 30 độ": img_rotated_30,
    "Tương tự - Làm mờ": img_blurred,
    "Tương tự - Nhiễu nặng": img_noisy_heavy,
    "Khác biệt 1 (anh1.jpg)": cv2.imread(os.path.join(img_dir, "anh1.jpg"), cv2.IMREAD_GRAYSCALE),
    "Khác biệt 2 (anh2.jpg)": cv2.imread(os.path.join(img_dir, "anh2.jpg"), cv2.IMREAD_GRAYSCALE),
    "Khác biệt 3 (anh3.jpg)": cv2.imread(os.path.join(img_dir, "anh3.jpg"), cv2.IMREAD_GRAYSCALE),
    "Khác biệt 4 (anh4.jpg)": cv2.imread(os.path.join(img_dir, "anh4.jpg"), cv2.IMREAD_GRAYSCALE),
    "Khác biệt 5 (anh5.jpg)": cv2.imread(os.path.join(img_dir, "anh5.jpg"), cv2.IMREAD_GRAYSCALE),
    "Khác biệt 6 (anh6.jpg)": cv2.imread(os.path.join(img_dir, "anh6.jpg"), cv2.IMREAD_GRAYSCALE),
    "Khác biệt 7 (anh7.jpg)": cv2.imread(os.path.join(img_dir, "anh7.jpg"), cv2.IMREAD_GRAYSCALE),
    "Khác biệt 8 (anh8.jpg)": cv2.imread(os.path.join(img_dir, "anh8.jpg"), cv2.IMREAD_GRAYSCALE),
    "Khác biệt 9 (anh9.jpg)": cv2.imread(os.path.join(img_dir, "anh9.jpg"), cv2.IMREAD_GRAYSCALE),
    "Khác biệt 10 (anh10.jpg)": cv2.imread(os.path.join(img_dir, "anh10.jpg"), cv2.IMREAD_GRAYSCALE),
    "Khác biệt 11 (anh11.jpg)": cv2.imread(os.path.join(img_dir, "anh11.jpg"), cv2.IMREAD_GRAYSCALE),
    "Khác biệt 12 (anh12.jpg)": cv2.imread(os.path.join(img_dir, "anh12.jpg"), cv2.IMREAD_GRAYSCALE),
    "Khác biệt 13 (anh13.jpg)": cv2.imread(os.path.join(img_dir, "anh13.jpg"), cv2.IMREAD_GRAYSCALE)
}

fig, axes = plt.subplots(4, 5, figsize=(18, 14))
axes = axes.ravel()
for i, (title, img) in enumerate(all_images.items()):
    if img is not None:
        axes[i].imshow(img, cmap='gray')
        axes[i].set_title(title, fontsize=9)
    axes[i].axis('off')
for j in range(len(all_images), len(axes)):
    axes[j].axis('off')
plt.tight_layout()
plt.show()

## 3. Định Nghĩa Thuật Toán Băm Wavelet Hashing

Chúng ta cài đặt cả phương pháp băm thô (Slide) và phương pháp băm nhận thức chuẩn (Standard wHash).

In [ ]:
def custom_quantize(c, step=2):
    if isinstance(c, tuple):
        return tuple(custom_quantize(sub_c, step) for sub_c in c)
    elif isinstance(c, np.ndarray):
        return np.round(c / step).astype(np.int64)
    else:
        return int(np.round(c / step))

def flatten_coeffs(coeffs):
    flat = []
    for c in coeffs:
        if isinstance(c, tuple):
            for sub_c in c: flat.append(sub_c.flatten())
        else:
            flat.append(c.flatten())
    return np.concatenate(flat)

def wavelet_hash_slide(image_path, wavelet='db4', level=1, step=2, size=(64, 64)):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    img_resized = cv2.resize(img, size)
    coeffs = pywt.wavedec2(img_resized.astype(np.float32), wavelet, level=level)
    coeffs_quant = [custom_quantize(c, step) for c in coeffs]
    flattened = flatten_coeffs(coeffs_quant)
    return [int(bit % 2) for bit in flattened]

def wavelet_hash_standard(image_path, wavelet='db4', level=1, size=(64, 64)):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    img_resized = cv2.resize(img, size)
    coeffs = pywt.wavedec2(img_resized.astype(np.float32), wavelet, level=level)
    cA = coeffs[0]
    med = np.median(cA)
    return (cA > med).astype(int).flatten().tolist()

In [ ]:
def hamming_distance(hash1, hash2):
    # Tối ưu hóa bằng NumPy (vectorization) thay thế vòng lặp Python
    return int(np.sum(np.abs(np.array(hash1) - np.array(hash2))))

image_keys = list(all_images.keys())
image_paths = {
    "Gốc (anh.jpg)": os.path.join(img_dir, "anh.jpg"),
    "Tương tự - Sáng hơn": os.path.join(img_dir, "anh_bright.jpg"),
    "Tương tự - Xoay 15 độ": os.path.join(img_dir, "anh_rotated.jpg"),
    "Tương tự - Nhiễu": os.path.join(img_dir, "anh_noisy.jpg"),
    "Tương tự - Xoay 30 độ": os.path.join(img_dir, "anh_rotated_30.jpg"),
    "Tương tự - Làm mờ": os.path.join(img_dir, "anh_blurred.jpg"),
    "Tương tự - Nhiễu nặng": os.path.join(img_dir, "anh_noisy_heavy.jpg"),
    "Khác biệt 1 (anh1.jpg)": os.path.join(img_dir, "anh1.jpg"),
    "Khác biệt 2 (anh2.jpg)": os.path.join(img_dir, "anh2.jpg"),
    "Khác biệt 3 (anh3.jpg)": os.path.join(img_dir, "anh3.jpg"),
    "Khác biệt 4 (anh4.jpg)": os.path.join(img_dir, "anh4.jpg"),
    "Khác biệt 5 (anh5.jpg)": os.path.join(img_dir, "anh5.jpg"),
    "Khác biệt 6 (anh6.jpg)": os.path.join(img_dir, "anh6.jpg"),
    "Khác biệt 7 (anh7.jpg)": os.path.join(img_dir, "anh7.jpg"),
    "Khác biệt 8 (anh8.jpg)": os.path.join(img_dir, "anh8.jpg"),
    "Khác biệt 9 (anh9.jpg)": os.path.join(img_dir, "anh9.jpg"),
    "Khác biệt 10 (anh10.jpg)": os.path.join(img_dir, "anh10.jpg"),
    "Khác biệt 11 (anh11.jpg)": os.path.join(img_dir, "anh11.jpg"),
    "Khác biệt 12 (anh12.jpg)": os.path.join(img_dir, "anh12.jpg"),
    "Khác biệt 13 (anh13.jpg)": os.path.join(img_dir, "anh13.jpg")
}

hashes_slide = {name: wavelet_hash_slide(path) for name, path in image_paths.items()}
hashes_std = {name: wavelet_hash_standard(path) for name, path in image_paths.items()}

pairs_results = []
print(f"{'='*95}")
print(f"{'Cặp Ảnh':<50} | {'Ground Truth':<12} | {'Khoảng Cách Slide':<15} | {'Khoảng Cách wHash Chuẩn':<15}")
print(f"{'='*95}")

for i in range(len(image_keys)):
    for j in range(i + 1, len(image_keys)):
        name1, name2 = image_keys[i], image_keys[j]
        is_similar = 1 if (i < 7 and j < 7) else 0
        dist_slide = hamming_distance(hashes_slide[name1], hashes_slide[name2])
        dist_std = hamming_distance(hashes_std[name1], hashes_std[name2])
        
        gt_label = "Tương tự" if is_similar == 1 else "Khác biệt"
        print(f"{name1} vs {name2:<50} | {gt_label:<12} | {dist_slide:<15} | {dist_std:<15}")
        
        pairs_results.append({
            "pair": [name1, name2],
            "gt": is_similar,
            "dist_slide": dist_slide,
            "dist_std": dist_std
        })
print(f"{'='*95}")

# Lưu trữ kết quả so khớp để notebook Phần 2 sử dụng
data_out_dir = "../data/processed"
os.makedirs(data_out_dir, exist_ok=True)
with open(os.path.join(data_out_dir, "pairs_results.json"), "w") as f:
    json.dump({
        "max_len_slide": len(hashes_slide["Gốc (anh.jpg)"]),
        "max_len_std": len(hashes_std["Gốc (anh.jpg)"]),
        "results": pairs_results
    }, f, indent=1)
print("Đã lưu kết quả phân tích sang data/processed/pairs_results.json!")